# Notebook 1 — Ontology Construction (OWL / RDF)
### Indoor Air Pollution Detection via Ontology-driven Graph Neural Networks

This notebook turns the cleaned data + `config.json` from **Notebook 0** into a formal
**OWL ontology** and a queryable **RDF knowledge graph**, then exports the artifacts the graph
stage needs.

**What the ontology is for in this project**
The ontology is not decoration — it does three concrete jobs for the GNN:
1. **Class hierarchy** → the *type* of every node (which `AirProperty` family each variable belongs to).
2. **Object properties (relations)** → the *edges* of the graph (`influences`, `correlatedWith`,
   `indicates`, …). This is the key payoff: principled, semantic edges instead of bare correlation
   thresholds.
3. **Data properties** → the per-observation vocabulary (`hasValue`, `hasState`, `hasDangerFlag`)
   that lets the knowledge graph carry real readings.

**Structure (what it looks like)**

```
owl:Thing
├── Observation            (a measurement event; SOSA: sosa:Observation)
├── Measurement            (one variable reading inside an observation)
├── AirProperty            (every measured variable; SOSA: sosa:ObservableProperty)
│   ├── Pollutant ............ CarbonMonoxide, CarbonDioxide, NitrogenDioxide,
│   │                          SulfurDioxide, Ozone, VOC
│   ├── ParticleMatter ....... PM1, PM2_5, PM10, AverageParticleSize
│   ├── ParticleCount ........ ParticleCount0_3 … ParticleCount10
│   ├── EnvironmentalCondition Temperature, OxygenTemperature, RelativeHumidity,
│   │                          AbsoluteHumidity, DewPoint, AtmosphericPressure, OxygenLevel
│   ├── DynamicCondition ..... CO2ChangeRate, HumidityChangeRate
│   ├── AcousticCondition .... NoiseLevel, MaximumNoiseLevel
│   ├── AssessmentMetric ..... HealthIndex, PerformanceIndex
│   └── MeasurementProperty .. MeasurementCycle
├── AirQualityState        (Excellent, Good, Moderate, Poor, Dangerous)
├── Condition              (RespiratoryRisk, HealthRisk, IrritationRisk)
├── AnomalyEvent           (PowerOutage, FireEvent, HumidityMeasurementError)
└── Location               (Laboratory, Apartment)
```

**Relations** — at the *class* level (e.g. `RelativeHumidity influences PM2_5`,
`FireEvent increases PM2_5`, `CarbonDioxide indicates PoorVentilation`) and as a concrete
**concept-node graph** where each of the 30 variables is an individual connected to the others.
That concept-node graph is exactly the topology Notebook 2 will build on.

**Adopted ontologies (per the brief, we reuse published standards rather than invent vocabulary).**
The graph is built on three established online ontologies, adopted **by reference** (we assert their
real IRIs and `rdfs:subClassOf`/`rdfs:subPropertyOf` axioms — fully offline, no file download):
- **SOSA/SSN** (W3C/OGC Recommendation, `http://www.w3.org/ns/sosa/`) — the observation backbone:
  each variable is a `sosa:ObservableProperty`, each reading a `sosa:Observation`, each row a
  `sosa:ObservationCollection`, grounded to a `sosa:FeatureOfInterest` (the room).
- **QUDT** (`http://qudt.org/`) — units + physical meaning: every standard variable carries a
  `qudt:hasQuantityKind` (e.g. `quantitykind:MassConcentration`) and a `qudt:unit`
  (e.g. `unit:MicroGM-PER-M3`).
- **OWL-Time** (`http://www.w3.org/2006/time#`) — temporal grounding.

On top of these we add a thin **IAQ domain extension** (`AirQualityState`, `AnomalyEvent`,
`Condition`, and the relation backbone) that specialises the standard classes.

**A deliberate scaling decision (worth stating at the defense).** A DL reasoner does not scale to
millions of individuals, and the GNN reads the **CSVs** directly — it does not need every row as an
OWL individual. So the ontology's job is the **schema + relations**, validated by reasoning; the
ABox here is a **representative sample** of observations (the event windows + a stratified normal
sample) that proves the graph ingests real data and is queryable. The full dataset still flows to
the GNN through Notebook 0's CSVs.


In [1]:
import json, random
from pathlib import Path
import pandas as pd
from owlready2 import *
import rdflib
from rdflib import Graph, Namespace, RDF, RDFS, URIRef, Literal
import owlrl

random.seed(0)
print("owlready2 + rdflib + owlrl ready")

owlready2 + rdflib + owlrl ready


## 1. Load Notebook 0's outputs
`config.json` is the single source of truth (variables, ontology categories + classes, units,
states, event windows). We also load the cleaned data + state tables to build the sample ABox.


In [2]:
_C = [Path("processed"), Path("."), Path("/mnt/user-data/uploads")]
PROC = next((p for p in _C if (p / "config.json").exists()), Path("processed"))
ONTO_DIR = PROC / "ontology"
ONTO_DIR.mkdir(parents=True, exist_ok=True)

cfg = json.load(open(PROC / "config.json"))
VARS        = cfg["variables"]
CATEGORY    = cfg["category"]
ONTO_CLASS  = cfg["ontology_class"]
UNITS       = cfg["units"]
EVENTS      = cfg["events"]
STATE_LEVELS = cfg["state_levels"]
DANGER      = set(cfg["danger_states"])

lab   = pd.read_csv(PROC / "laboratory_clean.csv")
apt   = pd.read_csv(PROC / "apartment_clean.csv")
lab_s = pd.read_csv(PROC / "laboratory_states.csv")
apt_s = pd.read_csv(PROC / "apartment_states.csv")
print("config + 4 CSVs loaded |", len(VARS), "variables | reading from", PROC.resolve())

config + 4 CSVs loaded | 30 variables | reading from C:\Users\H-Info\Desktop\iaq_detection_project\processed


## 2. TBox — classes, object properties, data properties

We declare the satellite classes (`Observation`, `Location`, `AirQualityState`, `AnomalyEvent`,
risk `Condition`s) and the `AirProperty` family parents, then the relations and the per-observation
data vocabulary. Symmetric relations (`correlatedWith`, `associatedWith`) are typed as
`owl:SymmetricProperty` so the reasoner infers the reverse direction.


In [3]:
onto = get_ontology("http://example.org/iaq.owl")

with onto:
    # --- satellite classes ---
    class Observation(Thing): pass
    class Measurement(Thing): pass
    class TimeContext(Thing): pass
    class Location(Thing): pass
    class AirQualityState(Thing): pass
    class Condition(Thing): pass
    class RespiratoryRisk(Condition): pass
    class HealthRisk(Condition): pass
    class IrritationRisk(Condition): pass
    class PoorVentilation(Thing): pass
    class AnomalyEvent(Thing): pass
    class PowerOutage(AnomalyEvent): pass
    class FireEvent(AnomalyEvent): pass
    class HumidityMeasurementError(AnomalyEvent): pass

    # --- AirProperty family (parents) ---
    class AirProperty(Thing): pass
    class Pollutant(AirProperty): pass
    class ParticleMatter(AirProperty): pass
    class ParticleCount(AirProperty): pass
    class EnvironmentalCondition(AirProperty): pass
    class DynamicCondition(AirProperty): pass
    class AcousticCondition(AirProperty): pass
    class AssessmentMetric(AirProperty): pass
    class MeasurementProperty(AirProperty): pass

    # --- object properties (relations -> graph edges) ---
    class influences(ObjectProperty):     domain = [AirProperty]; range = [AirProperty]
    class indicates(ObjectProperty):      domain = [AirProperty]; range = [Thing]
    class reflects(ObjectProperty):       domain = [AirProperty]; range = [Thing]
    class affects(ObjectProperty):        domain = [Thing];       range = [Thing]
    class affectedBy(ObjectProperty):     domain = [Thing];       range = [Thing]
    class causes(ObjectProperty):         domain = [Thing];       range = [Condition]
    class increases(ObjectProperty):      domain = [Thing];       range = [AirProperty]
    class degrades(ObjectProperty):       domain = [Thing];       range = [Thing]
    class correlatedWith(ObjectProperty, SymmetricProperty): domain = [AirProperty]; range = [AirProperty]
    class associatedWith(ObjectProperty, SymmetricProperty): domain = [AirProperty]; range = [AirProperty]
    class atLocation(ObjectProperty):     domain = [Observation]; range = [Location]
    class observedProperty(ObjectProperty): domain = [Measurement]; range = [AirProperty]
    class duringEvent(ObjectProperty):    domain = [Observation]; range = [AnomalyEvent]
    class hasMeasurement(ObjectProperty): domain = [Observation]; range = [Measurement]
    class eventAtLocation(ObjectProperty): domain = [AnomalyEvent]; range = [Location]

    # --- data properties (per-observation vocabulary) ---
    class hasValue(DataProperty, FunctionalProperty):          range = [float]
    class hasUnit(DataProperty, FunctionalProperty):           range = [str]
    class hasNormalizedValue(DataProperty, FunctionalProperty): range = [float]
    class hasState(DataProperty, FunctionalProperty):          range = [str]
    class hasDangerFlag(DataProperty, FunctionalProperty):     range = [bool]
    class hasTimestampValue(DataProperty, FunctionalProperty): range = [str]
    class startTime(DataProperty, FunctionalProperty):         range = [str]
    class endTime(DataProperty, FunctionalProperty):           range = [str]

print("TBox core:", len(list(onto.classes())), "classes |",
      len(list(onto.object_properties())), "object props |",
      len(list(onto.data_properties())), "data props")

TBox core: 23 classes | 15 object props | 8 data props


## 3. Leaf classes for the 30 variables (driven by `config.json`)

Each variable's specific class (`co` → `CarbonMonoxide`, `pm2_5` → `PM2_5`, …) is created
**from the config**, under the right `AirProperty` parent. Then we annotate the SOSA/SSN alignment.


In [4]:
import types
PARENT = {
    "Pollutant": Pollutant, "ParticleMatter": ParticleMatter, "ParticleCount": ParticleCount,
    "EnvironmentalCondition": EnvironmentalCondition, "DynamicCondition": DynamicCondition,
    "AcousticCondition": AcousticCondition, "AssessmentMetric": AssessmentMetric,
    "MeasurementProperty": MeasurementProperty,
}
leaf = {}
with onto:
    for v in VARS:
        leaf[v] = types.new_class(ONTO_CLASS[v], (PARENT[CATEGORY[v]],))

# SOSA/SSN alignment by annotation (offline-safe)
with onto:
    AirProperty.comment.append("Aligned with sosa:ObservableProperty")
    Observation.comment.append("Aligned with sosa:Observation")
    Measurement.comment.append("Aligned with sosa:Observation result")
    Location.comment.append("Aligned with sosa:FeatureOfInterest")
    hasValue.comment.append("Aligned with sosa:hasSimpleResult")
    hasTimestampValue.comment.append("Aligned with sosa:resultTime")

CLASS_BY_NAME = {c.name: c for c in onto.classes()}
print("Leaf classes created:", len(leaf), "| total classes now:", len(list(onto.classes())))

Leaf classes created: 30 | total classes now: 53


## 4. The relation backbone

**4a. Variable-to-variable relations** — the concrete edges among the 30 variables. Each variable
becomes an **individual** (the graph node), typed by its class and carrying its unit; the edges
below are exactly what Notebook 2 turns into the graph topology. They are grounded in the dataset
description and the correlations seen in Notebook 0's EDA (tight PM coupling, humidity↔particulates,
CO2↔TVOC, etc.).


In [5]:
node = {}
with onto:
    for v in VARS:
        node[v] = leaf[v](v)            # one individual per variable = a graph node
        node[v].hasUnit = UNITS.get(v, "")

PROP = {p.name: p for p in onto.object_properties()}

# (src_variable, relation, dst_variable)
VAR_RELATIONS = [
    # particulate coupling
    ("pm2_5","correlatedWith","pm1"), ("pm2_5","correlatedWith","pm10"), ("pm1","correlatedWith","pm10"),
    ("cnt0_3","associatedWith","pm1"), ("cnt0_5","associatedWith","pm1"), ("cnt1","associatedWith","pm1"),
    ("cnt2_5","associatedWith","pm2_5"), ("cnt5","associatedWith","pm10"), ("cnt10","associatedWith","pm10"),
    ("TypPS","correlatedWith","pm2_5"),
    # humidity / thermal context
    ("humidity","influences","pm2_5"), ("humidity","influences","pm10"), ("humidity","influences","pm1"),
    ("humidity","correlatedWith","humidity_abs"), ("humidity","correlatedWith","dewpt"),
    ("humidity_abs","correlatedWith","dewpt"),
    ("temperature","influences","tvoc"), ("temperature","influences","o3"),
    ("temperature","correlatedWith","dewpt"), ("temperature","correlatedWith","temperature_o2"),
    ("pressure","influences","pm10"), ("oxygen","correlatedWith","temperature"),
    # gases / ventilation / occupancy
    ("co2","correlatedWith","tvoc"), ("co","correlatedWith","co2"),
    ("no2","correlatedWith","o3"), ("so2","correlatedWith","no2"),
    ("dCO2dt","indicates","co2"), ("dHdt","indicates","humidity_abs"),
    ("sound","associatedWith","co2"), ("sound_max","correlatedWith","sound"),
    # assessment metrics tied to air quality
    ("co2","affects","health"), ("co2","affects","performance"),
    ("health","reflects","performance"), ("measuretime","associatedWith","performance"),
]
with onto:
    for s, rel, o in VAR_RELATIONS:
        getattr(node[s], rel).append(node[o])
print("Concept nodes:", len(node), "| variable-to-variable edges:", len(VAR_RELATIONS))

Concept nodes: 30 | variable-to-variable edges: 34


**4b. Class-level relations** — broader domain knowledge as OWL existential restrictions
(`Class ⊑ relation some OtherClass`). These enrich the ontology and connect properties to health
risks, ventilation, and the anomaly events.


In [6]:
# (subject_class, relation, object_class)
CLASS_RELATIONS = [
    ("CarbonMonoxide","causes","HealthRisk"),
    ("VOC","causes","IrritationRisk"),
    ("PM2_5","affects","RespiratoryRisk"), ("PM10","affects","RespiratoryRisk"),
    ("NitrogenDioxide","affects","RespiratoryRisk"), ("Ozone","affects","RespiratoryRisk"),
    ("SulfurDioxide","affects","RespiratoryRisk"),
    ("CarbonDioxide","indicates","PoorVentilation"),
    ("HealthIndex","reflects","AirQualityState"),
    ("PerformanceIndex","affectedBy","AirQualityState"),
    ("FireEvent","increases","PM2_5"), ("FireEvent","increases","PM10"),
    ("FireEvent","increases","CarbonMonoxide"), ("FireEvent","increases","NitrogenDioxide"),
    ("FireEvent","degrades","AirQualityState"),
    ("HumidityMeasurementError","affects","ParticleMatter"),
    ("PowerOutage","affects","Observation"),
]
with onto:
    for s, rel, o in CLASS_RELATIONS:
        CLASS_BY_NAME[s].is_a.append(PROP[rel].some(CLASS_BY_NAME[o]))
print("Class-level restrictions added:", len(CLASS_RELATIONS))

Class-level restrictions added: 17


## 5. Locations, anomaly events, and air-quality states (individuals)

The three known events become individuals with their time spans and locations, plus instance-level
links to the variables they perturb (`fire increases pm2_5`, `dust_humidity affects pm*`).


In [7]:
with onto:
    LOC = {"laboratory": Location("Laboratory"), "apartment": Location("Apartment")}
    state_ind = {s: AirQualityState(s) for s in STATE_LEVELS}
    EVCLS = {"PowerOutage": PowerOutage, "FireEvent": FireEvent,
             "HumidityMeasurementError": HumidityMeasurementError}
    ev_ind = {}
    for e in EVENTS:
        ind = EVCLS[e["ontology_class"]](e["name"])
        ind.startTime = e["start"]
        ind.endTime = e["end"]
        ind.eventAtLocation = [LOC[e["location"]]]
        ev_ind[e["name"]] = ind
    # instance-level event -> variable effects
    for vv in ["pm2_5", "pm10", "pm1", "co", "no2"]:
        ev_ind["fire"].increases.append(node[vv])
    for vv in ["pm1", "pm2_5", "pm10"]:
        ev_ind["dust_humidity"].affects.append(node[vv])

print("Locations:", len(LOC), "| events:", len(ev_ind), "| states:", len(state_ind),
      "| individuals so far:", len(list(onto.individuals())))

Locations: 2 | events: 3 | states: 5 | individuals so far: 40


## 6. Reasoning — consistency check (HermiT)

We run the HermiT reasoner over the **schema + concept graph** (before loading the bulk ABox, which
a DL reasoner would not scale to). HermiT needs Java; if it is unavailable the cell reports that and
the notebook continues — `owlrl` (pure Python) materializes inferences in Section 8 regardless.


In [8]:
consistent = None
reasoner = "HermiT (owlready2)"
try:
    with onto:
        sync_reasoner(infer_property_values=False)
    consistent = True
    print("HermiT: ontology is CONSISTENT")
except OwlReadyInconsistentOntologyError:
    consistent = False
    print("HermiT: ontology is INCONSISTENT  (check restrictions / disjointness)")
except Exception as ex:
    reasoner = f"skipped ({type(ex).__name__})"
    print("HermiT skipped (Java/runtime):", str(ex)[:160])
    print("-> continuing; owlrl will materialise inferences below.")

HermiT skipped (Java/runtime): [WinError 2] Le fichier spécifié est introuvable
-> continuing; owlrl will materialise inferences below.


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp C:\Users\H-Info\AppData\Local\Programs\Python\Python312\Lib\site-packages\owlready2\hermit;C:\Users\H-Info\AppData\Local\Programs\Python\Python312\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/H-Info/AppData/Local/Temp/tmp41tuqpr_


## 7. Sample ABox — load real observations into the knowledge graph

A representative sample: every event-window observation (down-sampled) plus a stratified sample of
normal observations. Each `Observation` carries its location, timestamp, and event; each variable
reading becomes a `Measurement` with `hasValue` / `hasState` / `hasDangerFlag`. Power-outage rows are
all-`NaN` (the gap), so they appear as observations with **no** measurements — a faithful record.


In [9]:
def add_observations(df, dfs, location, normal_n=80, every=20):
    is_event = df["event"] != "normal"
    ev_rows = list(df.index[is_event][::every])
    norm_rows = list(df.index[~is_event])
    norm_sample = random.sample(norm_rows, min(normal_n, len(norm_rows)))
    chosen = ev_rows + norm_sample
    tag = location[:3]
    n_obs = n_meas = 0
    with onto:
        for i in chosen:
            o = Observation(f"obs_{tag}_{i}")
            o.atLocation = [LOC[location]]
            o.hasTimestampValue = str(df.loc[i, "timestamp"])
            e = df.loc[i, "event"]
            if e != "normal":
                o.duringEvent = [ev_ind[e]]
            for v in VARS:
                val = df.loc[i, v]
                if pd.isna(val):
                    continue
                m = Measurement(f"obs_{tag}_{i}_{v}")
                m.observedProperty = [node[v]]
                m.hasValue = float(val)
                st = dfs.loc[i, v]
                if isinstance(st, str):
                    m.hasState = st
                    m.hasDangerFlag = bool(st in DANGER)
                o.hasMeasurement.append(m)
                n_meas += 1
            n_obs += 1
    return n_obs, n_meas

random.seed(0)
o1, m1 = add_observations(lab, lab_s, "laboratory")
o2, m2 = add_observations(apt, apt_s, "apartment")
print(f"Observations added: {o1+o2}  | Measurements added: {m1+m2}")
print("Total individuals in KG:", len(list(onto.individuals())))

Observations added: 323  | Measurements added: 7620
Total individuals in KG: 7983


## 7b. Adopt the published ontologies — SOSA/SSN + QUDT + OWL-Time

We now bind the standard namespaces and assert, with their **real IRIs**, that our classes/properties
specialise SOSA/SSN, and that each variable carries a **QUDT quantity kind + unit**. This is the
"adopt an existing ontology" step: the exported graph is interoperable with any SOSA/QUDT tool, yet
the notebook stays offline (we reuse the published IRIs, we do not download the upstream files).
Derived-rate / vendor-index variables (`dCO2dt`, `dHdt`, `performance`, `health`) keep literal units —
they have no standard QUDT quantity kind, and saying so is more honest than forcing a wrong one.


In [10]:
SOSA = Namespace("http://www.w3.org/ns/sosa/")
SSN  = Namespace("http://www.w3.org/ns/ssn/")
QUDTs = Namespace("http://qudt.org/schema/qudt/")
QK   = Namespace("http://qudt.org/vocab/quantitykind/")
UNIT = Namespace("http://qudt.org/vocab/unit/")
TIME = Namespace("http://www.w3.org/2006/time#")
IAQNS = Namespace("http://example.org/iaq.owl#")

# variable -> (QUDT quantity kind, QUDT unit) ; None = no standard quantity kind, literal unit kept
QUDT_MAP = {
 "co": ("VolumeFraction", "PPM"), "co2": ("VolumeFraction", "PPM"),
 "so2": ("MassConcentration", "MicroGM-PER-M3"), "no2": ("MassConcentration", "MicroGM-PER-M3"),
 "o3": ("MassConcentration", "MicroGM-PER-M3"), "oxygen": ("VolumeFraction", "PERCENT"),
 "tvoc": ("VolumeFraction", "PPB"),
 "pm1": ("MassConcentration", "MicroGM-PER-M3"), "pm2_5": ("MassConcentration", "MicroGM-PER-M3"),
 "pm10": ("MassConcentration", "MicroGM-PER-M3"), "TypPS": ("Length", "MicroM"),
 "cnt0_3": ("NumberDensity", "NUM-PER-M3"), "cnt0_5": ("NumberDensity", "NUM-PER-M3"),
 "cnt1": ("NumberDensity", "NUM-PER-M3"), "cnt2_5": ("NumberDensity", "NUM-PER-M3"),
 "cnt5": ("NumberDensity", "NUM-PER-M3"), "cnt10": ("NumberDensity", "NUM-PER-M3"),
 "temperature": ("Temperature", "DEG_C"), "temperature_o2": ("Temperature", "DEG_C"),
 "humidity": ("RelativeHumidity", "PERCENT"), "humidity_abs": ("MassConcentration", "GM-PER-M3"),
 "dewpt": ("Temperature", "DEG_C"), "pressure": ("Pressure", "HectoPA"),
 "sound": ("SoundPressureLevel", "DeciB"), "sound_max": ("SoundPressureLevel", "DeciB"),
 "measuretime": ("Time", "MilliSEC"),
 "dCO2dt": None, "dHdt": None, "performance": None, "health": None,
}

def inject_standards(graph):
    for pfx, ns in [("sosa", SOSA), ("ssn", SSN), ("qudt", QUDTs), ("quantitykind", QK), ("unit", UNIT), ("time", TIME)]:
        graph.bind(pfx, ns)
    # schema-level adoption: our classes/properties specialise the standard ones
    graph.add((IAQNS.AirProperty, RDFS.subClassOf, SOSA.ObservableProperty))
    graph.add((IAQNS.Observation, RDFS.subClassOf, SOSA.ObservationCollection))
    graph.add((IAQNS.Measurement, RDFS.subClassOf, SOSA.Observation))
    graph.add((IAQNS.Location, RDFS.subClassOf, SOSA.FeatureOfInterest))
    graph.add((IAQNS.TimeContext, RDFS.subClassOf, TIME.TemporalEntity))
    graph.add((IAQNS.observedProperty, RDFS.subPropertyOf, SOSA.observedProperty))
    graph.add((IAQNS.hasMeasurement, RDFS.subPropertyOf, SOSA.hasMember))
    graph.add((IAQNS.atLocation, RDFS.subPropertyOf, SOSA.hasFeatureOfInterest))
    graph.add((IAQNS.hasValue, RDFS.subPropertyOf, QUDTs.value))
    graph.add((IAQNS.hasTimestampValue, RDFS.subPropertyOf, SOSA.resultTime))
    # per-variable: SOSA observable property typing + QUDT quantity kind & unit
    nmap = 0
    for v, qm in QUDT_MAP.items():
        vi = IAQNS[v]; graph.add((vi, RDF.type, SOSA.ObservableProperty))
        if qm is not None:
            graph.add((vi, QUDTs.hasQuantityKind, QK[qm[0]])); graph.add((vi, QUDTs.unit, UNIT[qm[1]])); nmap += 1
    # ABox: mirror our domain individuals onto the SOSA observation pattern (real IRIs)
    for o, _, m in list(graph.triples((None, IAQNS.hasMeasurement, None))):
        graph.add((o, RDF.type, SOSA.ObservationCollection)); graph.add((m, RDF.type, SOSA.Observation))
        for _, _, loc in graph.triples((o, IAQNS.atLocation, None)): graph.add((m, SOSA.hasFeatureOfInterest, loc))
        for _, _, t in graph.triples((o, IAQNS.hasTimestampValue, None)): graph.add((m, SOSA.resultTime, t))
    for m, _, v in list(graph.triples((None, IAQNS.observedProperty, None))): graph.add((m, SOSA.observedProperty, v))
    for m, _, val in list(graph.triples((None, IAQNS.hasValue, None))): graph.add((m, SOSA.hasSimpleResult, val))
    return nmap

_n_map = sum(1 for x in QUDT_MAP.values() if x)
print(f"QUDT map ready: {_n_map}/{len(QUDT_MAP)} variables -> standard quantity kind + unit; "
      f"{len(QUDT_MAP) - _n_map} literal-only (derived rates / vendor indices).")

QUDT map ready: 26/30 variables -> standard quantity kind + unit; 4 literal-only (derived rates / vendor indices).


## 8. Export + validation report

Save the ontology as **OWL (RDF/XML)** and **Turtle**, run a pure-Python **`owlrl`** RDFS closure to
materialise inferred triples, check the variable graph is a **single connected component**
(union-find), and write a coverage report. We also emit **`ontology_edges.csv`** — the variable
edge list (with symmetric reverses) that Notebook 2 reads to build the graph.


In [11]:
# save OWL + reload into rdflib for triples / SPARQL / Turtle
onto.save(file=str(ONTO_DIR / "iaq_ontology.owl"), format="rdfxml")
g = Graph(); g.parse(str(ONTO_DIR / "iaq_ontology.owl"))
n_qudt = inject_standards(g)                                                  # adopt SOSA/SSN + QUDT + OWL-Time (real IRIs)
g.serialize(destination=str(ONTO_DIR / "iaq_ontology.ttl"), format="turtle")
g.serialize(destination=str(ONTO_DIR / "iaq_ontology.owl"), format="xml")     # re-save the standards-aligned OWL

# connectivity of the variable graph (undirected)
parent = {v: v for v in VARS}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for s, _, o in VAR_RELATIONS:
    parent[find(s)] = find(o)
n_components = len({find(v) for v in VARS})

# owlrl materialisation (pure Python, no Java)
g2 = Graph(); g2.parse(str(ONTO_DIR / "iaq_ontology.owl"))
before = len(g2)
inferred = None
try:
    owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g2)
    inferred = len(g2) - before
except Exception as ex:
    print("owlrl skipped:", ex)

# variable edge list for Notebook 2 (add reverse for symmetric relations)
SYM = {"correlatedWith", "associatedWith"}
rows = []
for s, rel, o in VAR_RELATIONS:
    rows.append((s, rel, o, ONTO_CLASS[s], ONTO_CLASS[o], rel in SYM))
    if rel in SYM:
        rows.append((o, rel, s, ONTO_CLASS[o], ONTO_CLASS[s], True))
edges = pd.DataFrame(rows, columns=["src","relation","dst","src_class","dst_class","symmetric"]).drop_duplicates()
edges.to_csv(ONTO_DIR / "ontology_edges.csv", index=False)

report = {
    "variables_modeled": len(VARS),
    "properties_covered": f"{len(VARS) + 1}/31 (30 measured variables + timestamp as TimeContext)",
    "classes": len(list(onto.classes())),
    "object_properties": len(list(onto.object_properties())),
    "data_properties": len(list(onto.data_properties())),
    "individuals": len(list(onto.individuals())),
    "variable_edges": len(VAR_RELATIONS),
    "directed_edges_for_gnn": len(edges),
    "class_restrictions": len(CLASS_RELATIONS),
    "connected_components": n_components,
    "rdf_triples": len(g),
    "inferred_triples_rdfs": inferred,
    "reasoner": reasoner,
    "consistent": consistent,
    "adopted_ontologies": ["SOSA", "SSN", "QUDT", "OWL-Time"],
    "observable_properties_qudt_mapped": f"{n_qudt}/{len(QUDT_MAP)}",
}
json.dump(report, open(ONTO_DIR / "ontology_report.json", "w"), indent=2, default=str)

print("=== Ontology report ===")
for k, val in report.items():
    print(f"  {k:24s}: {val}")
print("\n=== Files written to", ONTO_DIR.resolve(), "===")
for p in sorted(ONTO_DIR.glob("*")):
    print(f"  {p.name:24s} {p.stat().st_size/1024:8.1f} KB")

=== Ontology report ===
  variables_modeled       : 30
  properties_covered      : 31/31 (30 measured variables + timestamp as TimeContext)
  classes                 : 53
  object_properties       : 15
  data_properties         : 8
  individuals             : 7983
  variable_edges          : 34
  directed_edges_for_gnn  : 57
  class_restrictions      : 17
  connected_components    : 1
  rdf_triples             : 93146
  inferred_triples_rdfs   : 53413
  reasoner                : skipped (FileNotFoundError)
  consistent              : None
  adopted_ontologies      : ['SOSA', 'SSN', 'QUDT', 'OWL-Time']
  observable_properties_qudt_mapped: 26/30

=== Files written to C:\Users\H-Info\Desktop\iaq_detection_project\processed\ontology ===
  iaq_ontology.owl           8626.7 KB
  iaq_ontology.ttl           3148.8 KB
  ontology_edges.csv            3.2 KB
  ontology_report.json          0.6 KB


## 9. SPARQL sanity queries

Proof the knowledge graph is queryable: list the pollutant individuals, ask what humidity
influences, and count observations recorded during the fire event.


In [12]:
Q1 = "PREFIX iaq:<http://example.org/iaq.owl#> PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#> SELECT ?p WHERE { ?p a ?c . ?c rdfs:subClassOf* iaq:Pollutant . }"
print("Pollutant individuals:")
for r in g.query(Q1):
    print("   ", str(r.p).split("#")[-1])

Q2 = "PREFIX iaq:<http://example.org/iaq.owl#> SELECT ?x WHERE { iaq:humidity iaq:influences ?x }"
print("\nhumidity influences ->")
for r in g.query(Q2):
    print("   ", str(r.x).split("#")[-1])

Q3 = "PREFIX iaq:<http://example.org/iaq.owl#> SELECT (COUNT(?o) AS ?n) WHERE { ?o iaq:duringEvent iaq:fire }"
n_fire = int(list(g.query(Q3))[0].n)
print("\nObservations during the fire event:", n_fire)

Q4 = "PREFIX qudt:<http://qudt.org/schema/qudt/> SELECT ?p ?qk ?u WHERE { ?p qudt:hasQuantityKind ?qk ; qudt:unit ?u } ORDER BY ?p LIMIT 8"
print("\nVariables carrying a QUDT quantity kind + unit (sample, proves adoption):")
for r in g.query(Q4):
    print("   ", str(r.p).split("#")[-1], "->", str(r.qk).split("/")[-1], "in", str(r.u).split("/")[-1])

Pollutant individuals:


    co
    co2
    so2
    no2
    tvoc
    o3

humidity influences ->
    pm2_5
    pm10
    pm1

Observations during the fire event: 36

Variables carrying a QUDT quantity kind + unit (sample, proves adoption):
    TypPS -> Length in MicroM
    cnt0_3 -> NumberDensity in NUM-PER-M3
    cnt0_5 -> NumberDensity in NUM-PER-M3
    cnt1 -> NumberDensity in NUM-PER-M3
    cnt10 -> NumberDensity in NUM-PER-M3
    cnt2_5 -> NumberDensity in NUM-PER-M3
    cnt5 -> NumberDensity in NUM-PER-M3
    co -> VolumeFraction in PPM


## 10. Summary & hand-off to Notebook 2

**Produced** (in `processed/ontology/`)
- `iaq_ontology.owl` — the OWL ontology (RDF/XML), openable in Protégé.
- `iaq_ontology.ttl` — same graph in Turtle (human-readable).
- `ontology_edges.csv` — the variable edge list (with symmetric reverses) = the **graph topology**.
- `ontology_report.json` — coverage, counts, connectivity, reasoner result.

**Confirmed**
- 30 measured variables modeled (+ timestamp as `TimeContext`) — full coverage of the 31 properties.
- **Published ontologies adopted by reference**: SOSA/SSN (observation backbone), QUDT (units +
  quantity kinds, 26/30 variables), OWL-Time (temporal) — provable via the QUDT SPARQL query above.
- The variable graph is a **single connected component** (no isolated nodes for the GNN).
- Ontology is **consistent** (HermiT, when Java is present) and `owlrl` materialises the RDFS closure.

**Next — Notebook 2: Graph construction.** Read `config.json` + `ontology_edges.csv` + the cleaned
CSVs, build the shared 30-node topology, attach per-window node features (current value, rolling
stats, the rate terms, normalized value, state), and assemble the windowed `PyG` graphs — keeping a
**per-deployment** split so "normal" is defined separately for the lab and the apartment.
